# E2 — Treinamento e validação MGGP (5×2 e 5×3)

Notebook único: treina N modelos independentes por braço e valida cada aprovado nas 5 pistas,
em paralelo (um processo por tentativa, CPU a 100 %).

Porta do `TrainingMGGP_LOOP.ipynb` da IC, com os mesmos parâmetros do **T48**
(`nDelays [1,2,5,10,25,50]`, 7 termos, `maxHeight 6`, 300 gerações × 300 indivíduos,
MShooting → Free-Run) e a mesma regra de aprovação (`0 < RMSE(Wang 2.2, Free-Run) < 100`).

**Braços**
| braço | saídas | β |
|---|---|---|
| `5x2` | v_x, v_y | pós-calculado: `atan2(v̂y, v̂x)` |
| `5x3` | v_x, v_y, β | evoluído como 3ª saída (`atan2(col 17, col 18)`, em graus) |

**Regras da campanha** (por braço)
- Meta: `RUNS` modelos aprovados.
- Tentativa **falha** quando o Free-Run da Wang 2.2 explode (inf/NaN/≥ `GATE_MAX`) ou o processo quebra.
- **`MAX_CONSEC_FAIL` falhas seguidas ⇒ braço = FRACASSO**: nada mais é lançado e o que está rodando é cancelado.
- Uma aprovação **zera** o contador (não é acumulativo). "Seguidas" = na ordem em que as tentativas terminam.
- Nunca lança além da meta (em voo ≤ `RUNS − aprovados`).
- **Retomável**: pare o kernel e rode tudo de novo — ele continua do `attempts.csv`.

**Uso**: rode as células na ordem. A campanha é a célula 5; o resumo e os gráficos, a célula 6.
Acompanhe o progresso pelo print de cada tentativa e pelos logs em
`results/e2/<config>/<braço>/attempts/attempt_XXX/log.txt`.

## 1. Configuração

In [ ]:
# Tudo o que você normalmente mexe está nesta célula.
CONFIG            = "t48"        # "t48" (campeão da IC) ou "t19" (eleito, mais barato)
ARMS              = ["5x2", "5x3"]   # ordem de execução das campanhas
RUNS              = 30           # modelos aprovados por braço
WORKERS           = None         # None = todos os núcleos lógicos; ou um inteiro (ex.: 40)
MAX_CONSEC_FAIL   = 5            # falhas seguidas que decretam FRACASSO do braço
GATE_MAX          = 100.0        # aprova se 0 < RMSE(Wang 2.2, Free-Run) < GATE_MAX
BETA_UNIT         = "deg"        # unidade de β no treino do 5x3 — ver nota abaixo
SEED              = 2026         # tentativa k usa SEED + k (mesma semente nos dois braços)
OUT_DIR           = None         # None = results/e2/<CONFIG>
PAINEL_SEGUNDOS   = 5            # painel de progresso ao vivo: intervalo de atualização (0 desliga)
STALL_SEGUNDOS    = 120          # log parado por tanto tempo => o painel avisa "travou?"

# Overrides para teste rápido (deixe None na campanha de verdade):
GENERATIONS = None   # ex.: 3
POPULATION  = None   # ex.: 20
MAX_SAMPLES = None   # ex.: 1500 — trunca todas as pistas

# Nota sobre BETA_UNIT: a fitness da lib é a média simples das RMSE por saída, sem normalização.
# Em radianos (std ≈ 0,04) o β ficaria invisível ao lado de v_x (std ≈ 3,6 m/s), então o padrão é graus.

## 2. Constantes, dados e verificação prévia

In [ ]:
import json
import os
import shutil
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed, parallel_config


def _achar_raiz(inicio):
    """Raiz do repositório = primeira pasta (subindo) que contém src/mggp.py.
    Funciona com o Jupyter aberto em notebooks/, na raiz, ou em qualquer subpasta."""
    for d in [inicio, *inicio.parents]:
        if (d / "src" / "mggp.py").exists():
            return d
    raise RuntimeError(f"não achei a raiz do repositório a partir de {inicio} "
                       f"(esperava encontrar src/mggp.py subindo na árvore)")


ROOT = _achar_raiz(Path.cwd().resolve())
sys.path.insert(0, str(ROOT))          # a lib vendorizada usa imports absolutos (from src.mggp import ...)
DATA_DIR = ROOT / "Database"

TRACKS = {                              # nome -> (arquivo, papel)
    "Wang 2.1 (Treino)":    ("wang21dv_bic_MGGP.xlsx", "train"),
    "Wang 2.2 (Validação)": ("wang22dv_bic_MGGP.xlsx", "val"),
    "Wang 3.1 (Teste A)":   ("wang31dv_bic_MGGP.xlsx", "testA"),
    "Wang 8.1 (Teste B)":   ("wang81dv_bic_MGGP.xlsx", "testB"),
    "Wang 3.2 (Teste C)":   ("wang32dv_bic_MGGP.xlsx", "testC"),
}
GATE_TRACK  = "Wang 2.2 (Validação)"
INPUT_COLS  = [14, 2, 22, 23, 11]       # ax_nobias, acc_y, roda diant., roda tras., YawRate — como a IC
VX_COL, VY_COL = 18, 17                 # vx_smooth, vy_smooth  (β := atan2(col 17, col 18) — ver GOALS 1)

CONFIGS = {                             # âncoras da IC (parametros_utilizados.csv de cada treinamento)
    "t48": {"nDelays": [1, 2, 5, 10, 25, 50], "nTerms": 7, "maxHeight": 6},
    "t19": {"nDelays": [1, 2, 5, 10],         "nTerms": 3, "maxHeight": 5},
}
FIXED = {
    "generations": 300, "populationSize": 300, "evaluationMode": "RMSE", "k": 300,
    "evaluationType": "MShooting", "evaluationTypeTest": "FreeRun",
    "mutationRate": 0.3, "crossoverRate": 0.8, "elitePercentage": 10,
    "mode": "MIMO", "froe_mode": False, "operators": ["add", "subtraction", "mul"],
}
ATTEMPT_COLS = ["attempt", "seed", "status", "rmse_val", "seconds", "model", "started", "finished", "error"]


def build_params():
    p = {**FIXED, **CONFIGS[CONFIG]}
    if GENERATIONS:
        p["generations"] = GENERATIONS
    if POPULATION:
        p["populationSize"] = POPULATION
    return p


def load_track(file, arm, beta_unit=None, max_samples=None):
    """Mesmo loader do notebook da IC (`carregar_dados_bicicleta`) + β opcional como 3ª saída."""
    df = pd.read_excel(DATA_DIR / file, header=None)
    if max_samples:
        df = df.iloc[:max_samples]
    u = df.iloc[:, INPUT_COLS].to_numpy(dtype=float)
    vx, vy = df[VX_COL].to_numpy(dtype=float), df[VY_COL].to_numpy(dtype=float)
    beta = np.arctan2(vy, vx)
    if (beta_unit or BETA_UNIT) == "deg":
        beta = np.degrees(beta)
    y = np.column_stack([vx, vy, beta]) if arm == "5x3" else np.column_stack([vx, vy])
    return u, y


def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


def to_rad(x, beta_unit):
    return np.radians(x) if beta_unit == "deg" else x


def free_run(model, y, u, params):
    args = (y, u) if params["evaluationTypeTest"] != "MShooting" else (params["k"], y, u)
    yp, yd = model.predict(params["evaluationTypeTest"], *args)
    n = min(len(yp), len(yd))
    return yp[:n], yd[:n]


def agora():
    return datetime.now().isoformat(timespec="seconds")


def preflight():
    """Falha cedo e claro se faltar dado — senão cada tentativa quebra e o braço marca FRACASSO
    por falta de arquivo, não por instabilidade do modelo."""
    faltando = [f for f, _ in TRACKS.values() if not (DATA_DIR / f).exists()]
    if faltando:
        raise FileNotFoundError(
            f"dataset ausente em {DATA_DIR}\n  faltam {len(faltando)}/{len(TRACKS)}: {', '.join(faltando)}\n"
            f"  copie os 5 .xlsx da base Wang/Jilin para essa pasta (ver README, seção 1)."
        )
    load_track(TRACKS["Wang 2.1 (Treino)"][0], "5x3", "deg", 50)   # lê de verdade
    print(f"dataset OK: {len(TRACKS)} pistas em {DATA_DIR}")


WORKERS_EFETIVOS = WORKERS or os.cpu_count() or 1
preflight()
print(f"config {CONFIG} · braços {ARMS} · {RUNS} modelos/braço · {WORKERS_EFETIVOS} workers · β em {BETA_UNIT}")
print(f"parâmetros: {build_params()}")

## 3. Uma tentativa: treinar + validar

`treinar_e_validar` roda num processo separado (joblib/loky). Ela treina na Wang 2.1, aplica o gate
na Wang 2.2 e — se aprovar — valida Free-Run nas 5 pistas, salvando modelo (`dill`), predições
(`.npz`), figura e as linhas de RMSE dentro de `attempts/attempt_XXX/`. Todo o stdout do treinamento
(incluindo o log por geração da lib) vai para `log.txt` dessa pasta.

In [ ]:
def treinar_e_validar(job):
    """Executada num processo separado. Devolve um dicionário pequeno; os artefatos ficam em disco."""
    import contextlib
    import json
    import random
    import sys
    import time
    from pathlib import Path

    import dill
    import matplotlib
    import numpy as np
    import pandas as pd

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    sys.path.insert(0, str(job["root"]))
    from src.mggp import MGGP

    out = Path(job["out_dir"])
    out.mkdir(parents=True, exist_ok=True)
    arm, beta_unit, params = job["arm"], job["beta_unit"], job["params"]
    res = {"attempt": job["attempt"], "seed": job["seed"], "arm": arm,
           "approved": False, "rmse_val": None, "error": None}
    t0 = time.time()

    with open(out / "log.txt", "w", encoding="utf-8") as log, contextlib.redirect_stdout(log), \
            contextlib.redirect_stderr(log):
        try:
            random.seed(job["seed"])
            np.random.seed(job["seed"])
            print(f"=== {arm} tentativa {job['attempt']} seed {job['seed']} — {time.ctime()}")

            u_tr, y_tr = load_track(job["train_file"], arm, beta_unit, job["max_samples"])
            u_va, y_va = load_track(job["gate_file"], arm, beta_unit, job["max_samples"])
            mggp = MGGP(inputs=u_tr, outputs=y_tr, validation=(u_va, y_va),
                        filename=str(out / "temp_mggp.pkl"), **params)
            mggp.run()
            best = mggp._hof[0]

            # --- gate: Free-Run na pista de validação (regra da IC)
            try:
                yp, yd = free_run(best, y_va, u_va, params)
                err = float(best.score(yd, yp, params["evaluationMode"]))
            except Exception as e:
                err, res["error"] = float("inf"), f"gate: {type(e).__name__}: {e}"
            res["rmse_val"] = err if np.isfinite(err) else None
            res["approved"] = bool(np.isfinite(err) and 0.0 < err < job["gate_max"])
            print(f"gate {job['gate_file']}: RMSE {err} -> {'APROVADO' if res['approved'] else 'REPROVADO'}")

            if res["approved"]:
                with open(out / "model.pkl", "wb") as f:
                    dill.dump(best, f)
                (out / "equation.txt").write_text(f"{best}\ntheta:\n{best._theta}\n", encoding="utf-8")

                # --- validação Free-Run nas 5 pistas
                rows, preds = [], {}
                n_out = 3 if arm == "5x3" else 2
                fig, axes = plt.subplots(len(job["tracks"]), 3, figsize=(20, 4 * len(job["tracks"])))
                fig.suptitle(f"Free-Run — {out.name} ({arm}, beta em {beta_unit})", fontsize=15, y=1.0)

                for i, (track, file, role) in enumerate(job["tracks"]):
                    u, y = load_track(file, arm, beta_unit, job["max_samples"])
                    try:
                        yp, yd = free_run(best, y, u, params)
                        ok = bool(np.all(np.isfinite(yp)) and np.abs(yp).max() < 1e4)
                    except Exception as e:      # divergiu: registra e segue, como no notebook da IC
                        print(f"  ! {track}: {type(e).__name__}: {e}")
                        yp, yd, ok = np.zeros_like(y), y, False
                    preds[f"{role}_yd"], preds[f"{role}_yp"] = yd, yp

                    beta_true = to_rad(yd[:, 2], beta_unit) if n_out == 3 else np.arctan2(yd[:, 1], yd[:, 0])
                    beta_post = np.arctan2(yp[:, 1], yp[:, 0])   # beta do 5x2: atan2 das velocidades preditas
                    row = {"Pista": track, "Papel": role, "ok": ok,
                           "RMSE_Vx": rmse(yd[:, 0], yp[:, 0]) if ok else np.nan,
                           "RMSE_Vy": rmse(yd[:, 1], yp[:, 1]) if ok else np.nan,
                           "RMSE_beta_atan2_deg": np.degrees(rmse(beta_true, beta_post)) if ok else np.nan}
                    if n_out == 3:
                        beta_out = to_rad(yp[:, 2], beta_unit)   # beta evoluído (3a saída)
                        row["RMSE_beta_out_deg"] = np.degrees(rmse(beta_true, beta_out)) if ok else np.nan
                    row["RMSE_Medio_VxVy"] = (row["RMSE_Vx"] + row["RMSE_Vy"]) / 2
                    rows.append(row)

                    for j, (lbl, col) in enumerate([("Vx [m/s]", 0), ("Vy [m/s]", 1)]):
                        ax = axes[i, j]
                        ax.plot(yd[:, col], color="black", lw=0.8, label="Real")
                        ax.plot(yp[:, col], color="tab:blue" if j == 0 else "tab:red", lw=0.8,
                                ls="--", label="Previsto")
                        ax.set_title(f"{track} — {lbl} | RMSE {row['RMSE_Vx' if j == 0 else 'RMSE_Vy']:.4f}")
                        ax.grid(True, ls=":", alpha=0.6)
                    ax = axes[i, 2]
                    ax.plot(np.degrees(beta_true), color="black", lw=0.8, label="beta real")
                    ax.plot(np.degrees(beta_post), color="tab:green", lw=0.8, ls="--",
                            label=f"atan2(vy_hat, vx_hat) | {row['RMSE_beta_atan2_deg']:.3f} deg")
                    if n_out == 3:
                        ax.plot(np.degrees(beta_out), color="tab:purple", lw=0.8, ls=":",
                                label=f"saida 3 | {row['RMSE_beta_out_deg']:.3f} deg")
                    ax.set_title(f"{track} — beta [deg]")
                    ax.legend(fontsize=8)
                    ax.grid(True, ls=":", alpha=0.6)

                axes[0, 0].legend(fontsize=8)
                fig.tight_layout()
                fig.savefig(out / "fig.png", dpi=90, bbox_inches="tight")
                plt.close(fig)
                np.savez_compressed(out / "predictions.npz", **preds)
                pd.DataFrame(rows).to_csv(out / "validation.csv", index=False)
        except Exception as e:
            res["error"] = f"{type(e).__name__}: {e}"
            print(f"CRASH {res['error']}")
        finally:
            res["seconds"] = round(time.time() - t0)
            (out / "temp_mggp.pkl").unlink(missing_ok=True)
            (out / "result.json").write_text(json.dumps(res), encoding="utf-8")
            print(f"fim em {res['seconds']} s")
    return res

## 3b. Painel de progresso ao vivo

Uma thread no processo principal lê o fim do `log.txt` de cada tentativa em andamento (a lib grava uma linha
`gen evals min avg max` por geração) e redesenha a tabela abaixo a cada `PAINEL_SEGUNDOS`. `⚠` = o log parou de
crescer há mais de `STALL_SEGUNDOS` (provável travamento). Só lê arquivos: não altera a lib nem o treino.

In [ ]:
import re
import threading

from IPython.display import display

_LINHA_GEN = re.compile(r"^(\d+)\s+(\d+)\s+(\S+)\s+(\S+)\s+(\S+)$")   # logbook da lib: gen evals min avg max
_CABECALHO = (f" {'tent':>4}  {'seed':>5}   {'geração':<9}  {'progresso':<18} {'melhor':<9} {'média':<9} "
              f"{'tempo':<7} {'ETA':<7} último log")


def _fim_do_log(path, nbytes=65536):
    """Últimos `nbytes` do log (o arquivo inteiro passa de 1 MB por tentativa)."""
    with open(path, "rb") as f:
        f.seek(0, os.SEEK_END)
        f.seek(max(0, f.tell() - nbytes))
        return f.read().decode("utf-8", errors="replace")


def _ultima_geracao(txt):
    for linha in reversed(txt.splitlines()):
        m = _LINHA_GEN.match(linha.strip())
        if m:
            try:
                return int(m[1]), float(m[3]), float(m[4])
            except ValueError:
                continue
    return None, None, None


def _estado_tentativa(log):
    """(geração, melhor fitness, média, fase) lidos do fim do log."""
    for nbytes in (65536, 1 << 20):         # a barra tqdm da geração corrente pode empurrar a última linha para longe
        txt = _fim_do_log(log, nbytes)
        gen, melhor, media = _ultima_geracao(txt)
        if gen is not None or os.path.getsize(log) <= nbytes:
            break
    if "Executed in:" in txt:
        fase = "validando"          # treino acabou: gate + Free-Run nas 5 pistas (fica sem log por um tempo)
    elif gen is None:
        fase = "iniciando"          # lendo dados / avaliando a população inicial
    else:
        fase = "treinando"
    return gen, melhor, media, fase


def _mmss(seg):
    h, r = divmod(int(max(0, seg)), 3600)
    return f"{h}:{r // 60:02d}:{r % 60:02d}" if h else f"{r // 60:02d}:{r % 60:02d}"


class Painel:
    """Contexto que redesenha, em outra thread, a evolução de cada tentativa em andamento da leva."""

    def __init__(self, jobs, done, cabecalho, concluidas, total_gens):
        self.jobs, self.done, self.cabecalho = jobs, done, cabecalho
        self.concluidas, self.total = concluidas, total_gens
        self.visto = {}                                  # tentativa -> instante em que o log apareceu
        self.parar = threading.Event()
        self.handle = None
        self.thread = threading.Thread(target=self._loop, daemon=True)

    def __enter__(self):
        if PAINEL_SEGUNDOS:
            self.handle = display({"text/plain": "[painel] aguardando as primeiras gerações..."},
                                  raw=True, display_id=True)
            self.thread.start()
        return self

    def __exit__(self, *exc):
        if self.handle is not None:
            self.parar.set()
            self.thread.join(timeout=PAINEL_SEGUNDOS + 5)
            self._desenhar(final=True)
        return False

    def _loop(self):
        while True:
            self._desenhar()
            if self.parar.wait(PAINEL_SEGUNDOS):
                break

    def _desenhar(self, final=False):
        try:                                             # o painel nunca pode derrubar a campanha
            texto = self._texto(final)
        except Exception as e:
            texto = f"[painel] erro ao ler os logs: {type(e).__name__}: {e}"
        try:
            self.handle.update({"text/plain": texto}, raw=True)
        except Exception:
            pass

    def _linhas(self):
        agora_s, linhas, fila = time.time(), [], 0
        for j in self.jobs:
            k = j["attempt"]
            if k in self.done:
                continue
            log = Path(j["out_dir"]) / "log.txt"
            if not log.exists():
                fila += 1
                continue
            t0 = self.visto.setdefault(k, agora_s)
            gen, melhor, media, fase = _estado_tentativa(log)
            parado = agora_s - log.stat().st_mtime
            frac = 1.0 if fase == "validando" else min(gen / self.total, 1.0) if gen else 0.0
            n = round(frac * 11)
            barra = f"{'█' * n}{'░' * (11 - n)} {frac * 100:3.0f} %"
            eta = _mmss((agora_s - t0) * (1 - frac) / frac) if fase == "treinando" and frac > 0 else "--:--"
            if fase == "validando":
                ultimo = "validando as 5 pistas..."
            elif fase == "iniciando":
                ultimo = "lendo dados / população inicial"
            elif parado > STALL_SEGUNDOS:
                ultimo = f"⚠ {parado:.0f} s sem log (travou?)"
            else:
                ultimo = f"{parado:.0f} s"
            geracao = f"{gen}/{self.total}" if gen else "-"
            m_ = f"{melhor:.4f}" if melhor is not None else "-"
            a_ = f"{media:.4f}" if media is not None else "-"
            linhas.append(f" {k:>4}  {j['seed']:>5}   {geracao:<9}  {barra:<18} {m_:<9} {a_:<9} "
                          f"{_mmss(agora_s - t0):<7} {eta:<7} {ultimo}")
        return linhas, fila

    def _texto(self, final=False):
        linhas, fila = ([], 0) if final else self._linhas()
        topo = self.cabecalho() + ("  — leva encerrada" if final else f" · {len(linhas)} rodando · {fila} na fila")
        partes = [topo]
        if linhas:
            partes += ["", _CABECALHO, *linhas]
        if self.concluidas:
            partes += ["", "concluídas: " + " · ".join(self.concluidas[-8:])]
        return "\n".join(partes)


## 4. A campanha de um braço

Submete exatamente `RUNS - aprovados` tentativas por leva; o joblib mantém `WORKERS` rodando e as
demais na fila. Os resultados são consumidos **na ordem em que terminam**: ao bater a meta — ou as
`MAX_CONSEC_FAIL` falhas seguidas — a leva é interrompida e **o que estava rodando é cancelado**.

In [ ]:
def _ledger(arm_dir):
    csv = arm_dir / "attempts.csv"
    return (pd.read_csv(csv) if csv.exists() else pd.DataFrame(columns=ATTEMPT_COLS)), csv


def _falhas_no_fim(df):
    """Falhas consecutivas no fim do histórico (zera na última aprovação)."""
    n = 0
    for st in df.sort_values("finished")["status"][::-1]:
        if st in ("rejected", "crashed"):
            n += 1
        elif st == "approved":
            break
    return n


def rodar_campanha(arm, root):
    arm_dir = root / arm
    for sub in ("attempts", "models", "fig", "predictions"):
        (arm_dir / sub).mkdir(parents=True, exist_ok=True)
    df, csv = _ledger(arm_dir)

    registros = df.to_dict("records")
    aprovados = int((df["status"] == "approved").sum())
    consec = _falhas_no_fim(df)

    # Só entram no ledger tentativas que terminaram. As canceladas (FRACASSO, kernel morto) deixam
    # a pasta para trás: numeramos acima dela para não reaproveitar um diretório com log antigo.
    usadas = [int(d.name.removeprefix("attempt_")) for d in (arm_dir / "attempts").glob("attempt_*")
              if d.name.removeprefix("attempt_").isdigit()]
    prox = max([*usadas, int(df["attempt"].max()) if len(df) else 0]) + 1
    params = build_params()
    tracks = [(nome, arq, papel) for nome, (arq, papel) in TRACKS.items()]
    estado = "FRACASSO" if consec >= MAX_CONSEC_FAIL else ("DONE" if aprovados >= RUNS else "RUNNING")
    print(f"\n{'=' * 90}\n[{arm}] {estado} — {aprovados}/{RUNS} aprovados · {consec} falha(s) seguida(s) · "
          f"próxima tentativa {prox}\n{'=' * 90}")

    concluidas = []         # tentativas que terminaram nesta sessão (resumo do painel)

    def cabecalho():          # lê aprovados/consec no momento de cada redesenho
        return (f"[{arm}] {datetime.now():%H:%M:%S} · {aprovados}/{RUNS} aprovados · "
                f"{consec}/{MAX_CONSEC_FAIL} falha(s) seguida(s)")

    while estado == "RUNNING":
        faltam = RUNS - aprovados
        jobs = [{"root": str(ROOT), "arm": arm, "attempt": prox + i, "seed": SEED + prox + i,
                 "out_dir": str(arm_dir / "attempts" / f"attempt_{prox + i:03d}"),
                 "beta_unit": BETA_UNIT, "gate_max": GATE_MAX, "params": params, "tracks": tracks,
                 "train_file": TRACKS["Wang 2.1 (Treino)"][0], "gate_file": TRACKS[GATE_TRACK][0],
                 "max_samples": MAX_SAMPLES}
                for i in range(faltam)]
        prox += faltam
        done = set()            # tentativas da leva que já terminaram (o painel mostra as outras)

        print(f"[{arm}] leva de {faltam} tentativa(s) ({min(faltam, WORKERS_EFETIVOS)} em paralelo)...")

        with parallel_config(backend="loky", inner_max_num_threads=1), \
                Painel(jobs, done, cabecalho, concluidas, params["generations"]):
            resultados = Parallel(n_jobs=WORKERS_EFETIVOS, return_as="generator_unordered")(
                delayed(treinar_e_validar)(j) for j in jobs)
            for res in resultados:
                k = res["attempt"]
                done.add(k)
                adir = arm_dir / "attempts" / f"attempt_{k:03d}"
                if res["approved"]:
                    status = "approved"
                elif res["error"] and res["rmse_val"] is None and "gate" not in str(res["error"]):
                    status = "crashed"
                else:
                    status = "rejected"
                linha = {"attempt": k, "seed": res["seed"], "status": status, "rmse_val": res["rmse_val"],
                         "seconds": res["seconds"], "model": "", "started": "", "finished": agora(),
                         "error": res["error"] or ""}

                if status == "approved":
                    aprovados += 1
                    consec = 0
                    nome = f"modelo_rmse_{aprovados}"
                    linha["model"] = nome
                    shutil.move(adir / "model.pkl", arm_dir / "models" / f"{nome}.pkl")
                    shutil.move(adir / "fig.png", arm_dir / "fig" / f"{nome}.png")
                    shutil.move(adir / "predictions.npz", arm_dir / "predictions" / f"{nome}.npz")
                    val = pd.read_csv(adir / "validation.csv")
                    val.insert(0, "Modelo", nome)
                    rep_path = arm_dir / "relatorio_validacao_rmse.csv"
                    rep = pd.concat([pd.read_csv(rep_path), val], ignore_index=True) if rep_path.exists() else val
                    rep.to_csv(rep_path, index=False)
                    print(f"[{arm}] OK tentativa {k:03d} APROVADA -> {nome} · RMSE val {res['rmse_val']:.4f} · "
                          f"{res['seconds']} s · {aprovados}/{RUNS}")
                    concluidas.append(f"{k:03d} ✔ {res['rmse_val']:.4f} ({res['seconds'] // 60} min)")
                else:
                    consec += 1
                    print(f"[{arm}] X  tentativa {k:03d} {status.upper()} · RMSE val {res['rmse_val']} · "
                          f"falhas seguidas {consec}/{MAX_CONSEC_FAIL}")
                    concluidas.append(f"{k:03d} ✘ {status}")

                registros.append(linha)
                df = pd.DataFrame(registros, columns=ATTEMPT_COLS)
                df.to_csv(csv, index=False)

                if aprovados >= RUNS:
                    estado = "DONE"
                    break
                if consec >= MAX_CONSEC_FAIL:
                    estado = "FRACASSO"
                    print(f"[{arm}] !! FRACASSO: {consec} falhas seguidas — cancelando o que está rodando")
                    break
            del resultados      # cancela as tentativas pendentes/em execução da leva

    (arm_dir / "STATUS.txt").write_text(
        f"{estado}\narm={arm} aprovados={aprovados}/{RUNS} falhas_seguidas={consec} "
        f"tentativas={df['status'].value_counts().to_dict()}\natualizado={agora()}\n", encoding="utf-8")
    print(f"[{arm}] {estado} — {aprovados}/{RUNS} aprovados, {len(df)} tentativa(s) no total")
    return estado

## 5. Rodar as campanhas

In [ ]:
root = Path(OUT_DIR) if OUT_DIR else ROOT / "results" / "e2" / CONFIG
root.mkdir(parents=True, exist_ok=True)
(root / "params.json").write_text(json.dumps(
    {"config": CONFIG, "arms": ARMS, "runs": RUNS, "workers": WORKERS_EFETIVOS,
     "max_consecutive_fail": MAX_CONSEC_FAIL, "gate": GATE_TRACK, "gate_max": GATE_MAX,
     "beta_unit": BETA_UNIT, "beta_def": "atan2(col17, col18)", "input_cols": INPUT_COLS,
     "vx_col": VX_COL, "vy_col": VY_COL, "base_seed": SEED, "max_samples": MAX_SAMPLES,
     "params": build_params(), "lib": "src/ vendorizada @7514bfb", "inicio": agora()},
    indent=2, ensure_ascii=False), encoding="utf-8")
pd.DataFrame(list(build_params().items()), columns=["Parametro", "Valor"]).to_csv(
    root / "parametros_utilizados.csv", index=False)

t0 = time.time()
estados = {arm: rodar_campanha(arm, root) for arm in ARMS}
print(f"\n{'=' * 90}\nFIM — {estados} · {round((time.time() - t0) / 60, 1)} min\nresultados em {root}")

## 6. Resumo e gráficos

Média +- desvio entre os modelos aprovados, por pista (só modelos cujo Free-Run **não** explodiu na pista).
As predições completas ficam em `predictions/*.npz` — é delas que sai a análise do β depois.

In [ ]:
import matplotlib.pyplot as plt

for arm in ARMS:
    rep_path = root / arm / "relatorio_validacao_rmse.csv"
    if not rep_path.exists():
        print(f"[{arm}] sem resultados ainda")
        continue
    rep = pd.read_csv(rep_path)
    ok = rep[rep["ok"].astype(bool)]
    metricas = [c for c in rep.columns if c.startswith("RMSE_")]
    agg = ok.groupby("Pista")[metricas].agg(["mean", "std", "count"]).round(4)
    agg.to_csv(root / arm / "summary_agg.csv")
    print(f"\n=== {arm} — {ok['Modelo'].nunique()} modelo(s) · "
          f"{len(rep) - len(ok)} par(es) modelo×pista descartado(s) por explosão")
    print(agg.xs("mean", axis=1, level=1).to_string())

    fig, axes = plt.subplots(1, len(metricas), figsize=(4.2 * len(metricas), 4), squeeze=False)
    for ax, m in zip(axes[0], metricas, strict=True):
        dados = [ok.loc[ok["Pista"] == p, m].dropna().values for p in TRACKS]
        ax.boxplot(dados, tick_labels=[p.split()[1] for p in TRACKS])
        ax.set_title(m, fontsize=10)
        ax.grid(True, ls=":", alpha=0.6)
        ax.tick_params(axis="x", rotation=45)
    fig.suptitle(f"{arm} — dispersão entre modelos aprovados", y=1.02)
    fig.tight_layout()
    plt.show()